<a href="https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KundanKumar088/FlyRank-Internship-week1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [22]:
!git clone https://github.com/KundanKumar088/FlyRank-Internship-week1.git

fatal: destination path 'FlyRank-Internship-week1' already exists and is not an empty directory.


In [23]:
import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir("."))

for root, dirs, files in os.walk("."):
    if "content_refresh_anonymized.csv" in files:
        print("Found at:", os.path.join(root, "content_refresh_anonymized.csv"))

Current directory: /content
Files: ['.config', 'FlyRank-Internship-week1', 'sample_data']
Found at: ./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv


In [24]:
import pandas as pd

df = pd.read_csv("./FlyRank-Internship-week1/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [25]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [26]:
for col in df.columns:
    if "declin" in col.lower() or "trend" in col.lower() or "label" in col.lower():
        print(col)

trend_direction
trend_pct


In [27]:
# Create binary target
df["is_declining_label"] = (df["trend_direction"] == "declining").astype(int)

# Check class distribution
print(df["is_declining_label"].value_counts())

is_declining_label
0    30000
Name: count, dtype: int64


In [28]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Target
y = df["is_declining_label"]

# Remove target, leakage columns, and IDs
drop_cols = [
    "is_declining_label",
    "trend_direction",   # leakage
    "trend_pct",         # leakage
    "content_id",
    "client_id"
]

X = df.drop(columns=drop_cols)

# Missing-value flags
for col in ["word_count"]:
    if col in X.columns:
        X[col + "_missing"] = X[col].isna().astype(int)

# Numeric & categorical columns
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "category"]).columns

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# Final feature matrix
X_features = preprocessor.fit_transform(X)

print("Target shape:", y.shape)
print("Feature matrix shape:", X_features.shape)

Target shape: (30000,)
Feature matrix shape: (30000, 72)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

##Feature Notes

* **Search features** (`search_volume`, `competition`, `cpc`): Describe keyword demand. Missing values filled with the median. Available before prediction.
* **Content features** (`content_type`, `main_intent`, `word_count`, `char_count`): Describe content. Categorical features are one-hot encoded; numeric missing values use median imputation. Available before prediction.
* **Performance features** (`impressions_90d`, `clicks_90d`, `sessions_90d`, `ctr`, `engagement_rate`, `avg_position`): Represent historical performance. Missing values filled with the median. Available before prediction.
* **Freshness features** (`content_age_days`, `days_since_last_update`, `age_tier`, `freshness_tier`): Describe content age and freshness. Median imputation for numeric values and one-hot encoding for categorical values. Available before prediction.
* **Excluded features:** `trend_direction` and `trend_pct` (label leakage), `content_id` and `client_id` (IDs only).


In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage checks:

Removed trend_direction and trend_pct because they directly define the target.
Removed content_id and client_id since they are identifiers, not predictive features.
Used only historical features (90-day and previous-period metrics) that exist before prediction.
No future information or product flags were included.

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leakage_cols = ["trend_direction", "trend_pct", "content_id", "client_id"]

print("Leakage columns present:")
print([col for col in leakage_cols if col in X.columns])


Leakage columns present:
[]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*



* **`content_id`** – Unique identifier; not useful for prediction.
* **`client_id`** – Identifier only; used for grouping, not as a feature.
* **`trend_direction`** – Used to create the target label, so it causes data leakage.
* **`trend_pct`** – Directly determines the target label, so it causes data leakage.


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.